# 05. End-to-End Fine-Tuning

Unfreezes the backbone representations and applies differential learning rates (low backbone LR e.g. 1e-5, higher classification head LR e.g. 1e-4) with early stopping and checkpoint recovery.

In [ ]:
import torch
import torch.nn as nn
from cv_benchmark.utils import set_seed, get_device
from cv_benchmark.data import get_dataloaders
from cv_benchmark.models import create_model
from cv_benchmark.models.factory import get_parameter_groups
from cv_benchmark.training import Trainer
from cv_benchmark.evaluation import evaluate_model

set_seed(42)
device = get_device("cuda")

model_name = "resnet50"
loaders = get_dataloaders(model_name=model_name, batch_size=64, image_size=(224, 224))
model, head_names = create_model(model_name, num_classes=100, pretrained=True, mode="fine_tuning")

param_groups = get_parameter_groups(model, head_names, backbone_lr=1e-5, head_lr=1e-4)
optimizer = torch.optim.AdamW(param_groups)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

trainer = Trainer(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    experiment_name=f"{model_name}_finetune",
)
history = trainer.fit(epochs=15)
metrics = evaluate_model(model, loaders["test"], device=device, num_classes=100)
print(f"Fine-Tuned {model_name} Top-1 Accuracy: {metrics['top1_accuracy']}%")